# 観測予測モデル（自己回帰・マスク予測・拡散）

観測予測では「どの条件で次を当てるか」の設計が重要です。自己回帰・マスク予測・拡散は同じ予測問題に対して異なる情報の使い方をします。


## 背景と目的

モデル選択を誤ると、短期予測は当たっても長期ロールアウトで崩れます。

三つの予測方式を同じ系列データで比較できると、タスクに合わせた設計判断がしやすくなります。

ここでは1次元系列を使い、各方式の誤差特性を比較します。


## 最初に解きたい疑問

1. `自己回帰`、`マスク予測`、`拡散` は、どの情報の使い方が違うのか。
2. なぜ短期で当たっても長期ロールアウトで崩れるのか。
3. `mask_idx` の穴埋めは、なぜ前後平均なのか。
4. `den` を何回も平滑化するのが、なぜ拡散風なのか。
5. 3つの `MSE` は同じ尺度で比べてよいのか。


## 先に押さえる言葉

- `自己回帰`: 直前までの値から次の値を順番に予測する方法です。逐次予測に向いています。
- `マスク予測`: 一部を隠して、その穴を周辺情報から埋める方法です。欠損補完に向いています。
- `拡散`: ノイズを少しずつ取り除きながら元データに戻す考え方です。反復的に復元します。
- `ノイズ耐性`: ノイズが入っても予測が大きく崩れにくい性質です。実データでは重要になります。


## 実行前の見取り図

1. `AR MSE` が出て、自己回帰の誤差が見えるか。
2. `Mask-fill MSE` が出て、欠損補完の誤差が見えるか。
3. `Diffusion-like` の値が出て、反復復元の誤差が見えるか。


## 出力の読み方

- `AR MSE`、`Mask-fill MSE`、`Diffusion-like` を同列に読んでよいかの注意。
- `mask_idx` の穴埋め結果をどう解釈するか。


## つまずきやすい点

- 3方式が「何を入力として使えるか」の違いで分かれている説明。
- 拡散型の計算コストが高い理由の補足。
- 3方式が入力の使い方で分かれている説明。
- 拡散型の反復計算コストの理由。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- ここでの3方式比較は定性的な入口であり、厳密なベンチマーク比較ではない点。


## コード 1: 入力データを作る

このセルでは 入力データを作る ための最小コードを動かします。 実行時は「`AR MSE` が出て、自己回帰の誤差が見えるか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
import numpy as np
np.random.seed(1)

T = 80
t = np.arange(T)
series = np.sin(t / 6.0) + 0.1 * np.random.randn(T)


## コード 2: 入力データを作る

このセルでは 入力データを作る ための最小コードを動かします。 実行時は「`Mask-fill MSE` が出て、欠損補完の誤差が見えるか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
# 1) 自己回帰: x_t = a*x_{t-1} + b
x = series[:-1]
y = series[1:]
a = np.dot(x, y) / np.dot(x, x)
b = y.mean() - a * x.mean()
ar_pred = a * x + b
ar_mse = np.mean((ar_pred - y) ** 2)

# 2) マスク予測: 中央点を前後平均で補完
mask_idx = np.arange(1, T - 1, 5)
masked = series.copy()
masked[mask_idx] = np.nan
filled = masked.copy()
for i in mask_idx:
    filled[i] = 0.5 * (masked[i - 1] + masked[i + 1])
mask_mse = np.mean((filled[mask_idx] - series[mask_idx]) ** 2)

# 3) 拡散風予測: ノイズ付加 -> 逐次平滑で復元
noisy = series + 0.35 * np.random.randn(T)
den = noisy.copy()
for _ in range(20):
    den[1:-1] = 0.25 * den[:-2] + 0.5 * den[1:-1] + 0.25 * den[2:]
diff_mse = np.mean((den - series) ** 2)

print('AR MSE         =', round(ar_mse, 6))
print('Mask-fill MSE  =', round(mask_mse, 6))
print('Diffusion-like =', round(diff_mse, 6))


自己回帰は逐次予測に強く、マスク予測は欠損補完に強い設計です。拡散型はノイズ耐性が高い一方で反復計算コストが増えます。ここで並べた 3 つの MSE は、予測タスクと評価条件がそろった厳密比較ではなく、入力の使い方の違いを示す定性的な例として読んでください。
